In [ ]:
import torch

print("PyTorch version:", torch.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

    gpu_memory = torch.cuda.get_device_properties(0).total_memory
    print("GPU memory:", round(gpu_memory / (1024**3), 2), "GB")
else:
    print("GPU is not available. Check Runtime settings.")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU name: Tesla T4
GPU memory: 14.56 GB


In [ ]:
!nvidia-smi

Thu Nov 26 05:50:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from pathlib import Path

# Create PolyGen working directories
base = Path("/content/PolyGen")

(base / "third_party").mkdir(parents=True, exist_ok=True)
(base / "outputs").mkdir(parents=True, exist_ok=True)

print("PolyGen folders created!")

PolyGen folders created!


In [ ]:
%cd /content/PolyGen

!git clone https://github.com/VAST-AI-Research/TripoSR.git third_party/TripoSR

/content/PolyGen
Cloning into 'third_party/TripoSR'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 164 (delta 66), reused 45 (delta 45), pack-reused 66 (from 1)
Receiving objects: 100% (164/164), 36.71 MiB | 19.17 MiB/s, done.
Resolving deltas: 100% (68/68), done.


In [ ]:
%cd /content/PolyGen/third_party/TripoSR

!ls

/content/PolyGen/third_party/TripoSR
examples  gradio_app.py  README.md	   run.py
figures   LICENSE	 requirements.txt  tsr


In [ ]:
!cat README.md


# TripoSR

<a href="https://www.tripo3d.ai/research/triposr"><img src="https://img.shields.io/badge/Technical_Blog-gray?logo=

In [ ]:
%cd /content/PolyGen/third_party/TripoSR

!cat requirements.txt

/content/PolyGen/third_party/TripoSR
omegaconf==2.3.0
Pillow==10.1.0
einops==0.7.0
git+https://github.com/tatsy/torchmcubes.git
transformers==4.35.0
trimesh==4.0.5
rembg
huggingface-hub
imageio[ffmpeg]
gradio
xatlas==0.0.9
moderngl==5.10.0


In [ ]:
!grep -nEi "install|requirements|torch|torchmcubes|cuda|run.py" README.md

30:### Installation
32:- Install CUDA if available
33:- Install PyTorch according to your platform: [https://pytorch.org/get-started/locally/](https://pytorch.org/get-started/locally/) **[Please make sure that the locally-installed CUDA major version matches the PyTorch-shipped CUDA major version. For example if you have CUDA 11.x installed, make sure to install PyTorch compiled with CUDA 11.x.]**
34:- Update setuptools by `pip install --upgrade setuptools`
35:- Install other dependencies by `pip install -r requirements.txt`
39:python run.py examples/chair.png --output-dir output/
45:For detailed usage of this script, use `python run.py --help`.
53:> AttributeError: module 'torchmcubes_module' has no attribute 'mcubes_cuda'
57:> torchmcubes was not compiled with CUDA support, use CPU version instead.
59:This is because `torchmcubes` is compiled without CUDA support. Please make sure that 
61:- The locally-installed CUDA major version matches the PyTorch-shipped CUDA major version. For 

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA available: True
CUDA version: 13.0
GPU: Tesla T4


In [ ]:
%cd /content/PolyGen/third_party/TripoSR

!cat requirements.txt

/content/PolyGen/third_party/TripoSR
omegaconf==2.3.0
Pillow==10.1.0
einops==0.7.0
git+https://github.com/tatsy/torchmcubes.git
transformers==4.35.0
trimesh==4.0.5
rembg
huggingface-hub
imageio[ffmpeg]
gradio
xatlas==0.0.9
moderngl==5.10.0


In [ ]:
!python run.py --help

Traceback (most recent call last):
  File "/content/PolyGen/third_party/TripoSR/run.py", line 7, in <module>
    import rembg
ModuleNotFoundError: No module named 'rembg'


In [ ]:
%cd /content/PolyGen/third_party/TripoSR

/content/PolyGen/third_party/TripoSR


In [ ]:
!pip install --upgrade setuptools wheel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.3/876.3 kB 13.9 MB/s  0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.10.2
    Uninstalling setuptools-80.10.2:
      Successfully uninstalled setuptools-80.10.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
!pip install --upgrade setuptools wheel

In [ ]:
import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
from pathlib import Path
import subprocess

base = Path("/content/PolyGen")

(base / "third_party").mkdir(parents=True, exist_ok=True)
(base / "outputs").mkdir(parents=True, exist_ok=True)

repo = base / "third_party" / "TripoSR"

if not (repo / ".git").exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/VAST-AI-Research/TripoSR.git",
        str(repo)
    ], check=True)

print("PolyGen environment recreated!")
print("TripoSR folder:", repo)

PolyGen environment recreated!
TripoSR folder: /content/PolyGen/third_party/TripoSR


In [ ]:
%cd /content/PolyGen/third_party/TripoSR

!pip install -r requirements.txt

/content/PolyGen/third_party/TripoSR
  Cloning https://github.com/tatsy/torchmcubes.git to /tmp/pip-req-build-6fojx3tf
  Running command git clone --filter=blob:none --quiet https://github.com/tatsy/torchmcubes.git /tmp/pip-req-build-6fojx3tf
  Resolved https://github.com/tatsy/torchmcubes.git to commit a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> from git+https://github.com/tatsy/torchmcubes.git

note: This is an issue with the package mentioned above, not pip.
hint: See above for det

In [ ]:
%cd /content/PolyGen/third_party/TripoSR

!python -m pip install -v --no-build-isolation "git+https://github.com/tatsy/torchmcubes.git"

/content/PolyGen/third_party/TripoSR
Using pip 26.3 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
  Cloning https://github.com/tatsy/torchmcubes.git to /tmp/pip-req-build-d3iiwdqe
  Running command git version
  git version 2.43.0
  Running command git clone --filter=blob:none https://github.com/tatsy/torchmcubes.git /tmp/pip-req-build-d3iiwdqe
  Cloning into '/tmp/pip-req-build-d3iiwdqe'...
  Running command git rev-parse HEAD
  a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Resolved https://github.com/tatsy/torchmcubes.git to commit a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Running command git rev-parse HEAD
  a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Running command Preparing metadata (pyproject.toml)
  2026-11-26 06:20:45,739 - scikit_build_core - INFO - RUN: /usr/local/lib/python3.13/dist-packages/cmake/data/bin/cmake -E capabilities
  2026-11-26 06:20:45,789 - scikit_build_core - INFO - CMake version: 3.31.10
  *** scikit-build-core 1.1.2 using CMake 3.31.10 (metad

In [ ]:
%pip install --upgrade --no-cache-dir pybind11 cmake ninja scikit-build-core


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.2/30.2 MB 85.7 MB/s  0:00:00
  Attempting uninstall: cmake
    Found existing installation: cmake 3.31.10
    Uninstalling cmake-3.31.10:
      Successfully uninstalled cmake-3.31.10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [cmake]


In [ ]:
%pip install --upgrade --no-cache-dir pip setuptools wheel
%pip install --upgrade --no-cache-dir scikit-build-core


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.0 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.2.2
    Uninstalling pip-26.2.2:
      Successfully uninstalled pip-26.2.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scikit-build-core]


In [ ]:
import scikit_build_core.build
print("scikit-build-core is working")


scikit-build-core is working


In [ ]:
import os
import pybind11

# Tell CMake where pybind11 is installed
os.environ["CMAKE_PREFIX_PATH"] = pybind11.get_cmake_dir()

print(pybind11.get_cmake_dir())


/usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11


In [ ]:
%pip install --no-build-isolation --no-cache-dir torchmcubes


ERROR: Could not find a version that satisfies the requirement torchmcubes (from versions: none)
ERROR: No matching distribution found for torchmcubes


In [ ]:
%pip install --no-build-isolation --no-cache-dir \
  -Ccmake.define.CMAKE_PREFIX_PATH=$(python -c "import pybind11; print(pybind11.get_cmake_dir())") \
  torchmcubes


ERROR: Could not find a version that satisfies the requirement torchmcubes (from versions: none)
ERROR: No matching distribution found for torchmcubes


In [ ]:
%pip install --upgrade pip setuptools wheel scikit-build-core cmake ninja

In [ ]:
%pip install --no-build-isolation PACKAGE_NAME

  Preparing metadata (pyproject.toml) ... done
  Created wheel for PACKAGE_NAME: filename=package_name-0.1-py3-none-any.whl size=1242 sha256=0fafedabb5c47c161c26904ca6588b2c4f2f0dfa0dd7ccbf4b64ba3c504975b7
  Stored in directory: /root/.cache/pip/wheels/a1/0f/ab/19fecd3958a04b6fe3fb0d105f9e7d4182876db8c44cb1f1fc
Successfully built PACKAGE_NAME


In [1]:
%cd /content/PolyGen/third_party/TripoSR

!python -m pip install -v --no-build-isolation "git+https://github.com/tatsy/torchmcubes.git"

[Errno 2] No such file or directory: '/content/PolyGen/third_party/TripoSR'
/content
Using pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
  Cloning https://github.com/tatsy/torchmcubes.git to /tmp/pip-req-build-4nx_js1_
  Running command git version
  git version 2.43.0
  Running command git clone --filter=blob:none https://github.com/tatsy/torchmcubes.git /tmp/pip-req-build-4nx_js1_
  Cloning into '/tmp/pip-req-build-4nx_js1_'...
  Running command git rev-parse HEAD
  a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Resolved https://github.com/tatsy/torchmcubes.git to commit a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Running command git rev-parse HEAD
  a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Running command Preparing metadata (pyproject.toml)
  Preparing metadata (pyproject.toml) ... done
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 109, in _run_wrapper
    status 

In [2]:
!pip install -U scikit-build-core pybind11 cmake ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.2/30.2 MB 81.1 MB/s  0:00:00
  Attempting uninstall: cmake
    Found existing installation: cmake 3.31.10
    Uninstalling cmake-3.31.10:
      Successfully uninstalled cmake-3.31.10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-build-core]


In [3]:
import scikit_build_core
import torch

print("Build backend:", scikit_build_core.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Build backend: 1.1.2
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [4]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [5]:
%cd /content/PolyGen/third_party/TripoSR

!python -m pip install --no-build-isolation -v "git+https://github.com/tatsy/torchmcubes.git"

[Errno 2] No such file or directory: '/content/PolyGen/third_party/TripoSR'
/content
Using pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
  Cloning https://github.com/tatsy/torchmcubes.git to /tmp/pip-req-build-hzvsz_wt
  Running command git version
  git version 2.43.0
  Running command git clone --filter=blob:none https://github.com/tatsy/torchmcubes.git /tmp/pip-req-build-hzvsz_wt
  Cloning into '/tmp/pip-req-build-hzvsz_wt'...
  Running command git rev-parse HEAD
  a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Resolved https://github.com/tatsy/torchmcubes.git to commit a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Running command git rev-parse HEAD
  a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Running command Preparing metadata (pyproject.toml)
  2026-11-26 16:58:53,825 - scikit_build_core - INFO - RUN: /usr/local/lib/python3.13/dist-packages/cmake/data/bin/cmake -E capabilities
  2026-11-26 16:58:53,833 - scikit_build_core - INFO - CMake version: 4.4.5
  *** sc

In [6]:
import importlib.util

print("torchmcubes found:",
      importlib.util.find_spec("torchmcubes") is not None)

torchmcubes found: True


In [7]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [26]:
from pathlib import Path

# Define absolute paths to avoid working directory mismatches
triposr_dir = Path("/content/PolyGen/third_party/TripoSR")
requirements_path = triposr_dir / "requirements.txt"

if not requirements_path.exists():
    print(f"Error: {requirements_path} not found. Checking if git repository is cloned...")
    # Try to clone or locate it
else:
    lines = requirements_path.read_text().splitlines()

    # Exclude the torchmcubes GitHub dependency
    remaining = [
        line for line in lines
        if "torchmcubes" not in line.lower()
    ]

    output_path = triposr_dir / "requirements_remaining.txt"
    output_path.write_text("\n".join(remaining) + "\n")

    print(f"Remaining requirements saved to {output_path}:")
    print("\n".join(remaining))

Remaining requirements saved to /content/PolyGen/third_party/TripoSR/requirements_remaining.txt:
omegaconf==2.3.0
Pillow==10.1.0
einops==0.7.0
transformers==4.35.0
trimesh==4.0.5
rembg
huggingface-hub
imageio[ffmpeg]
gradio
xatlas==0.0.9
moderngl==5.10.0


In [9]:
%cd /content/PolyGen/third_party/TripoSR

from pathlib import Path

requirements = Path("requirements.txt")

lines = requirements.read_text().splitlines()

# Exclude the torchmcubes GitHub dependency
remaining = [
    line for line in lines
    if "torchmcubes" not in line.lower()
]

Path("requirements_remaining.txt").write_text(
    "\n".join(remaining) + "\n"
)

print("Remaining requirements:")
print("\n".join(remaining))

[Errno 2] No such file or directory: '/content/PolyGen/third_party/TripoSR'
/content


FileNotFoundError: [Errno 2] No such file or directory: 'requirements.txt'

In [10]:
from pathlib import Path
import subprocess

# Define PolyGen directories
base = Path("/content/PolyGen")
third_party = base / "third_party"
outputs = base / "outputs"
repo = third_party / "TripoSR"

# Recreate folders
third_party.mkdir(parents=True, exist_ok=True)
outputs.mkdir(parents=True, exist_ok=True)

# Clone TripoSR only if it is missing
if (repo / "requirements.txt").exists():
    print("TripoSR is already available!")
else:
    if repo.exists() and any(repo.iterdir()):
        raise RuntimeError(
            "TripoSR folder exists but looks incomplete. "
            "Please share the folder contents."
        )

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/VAST-AI-Research/TripoSR.git",
            str(repo)
        ],
        check=True
    )

# Verify
print("\nPolyGen directory:", base)
print("TripoSR available:", (repo / "run.py").exists())
print("Requirements available:", (repo / "requirements.txt").exists())
print("Outputs directory:", outputs.exists())


PolyGen directory: /content/PolyGen
TripoSR available: True
Requirements available: True
Outputs directory: True


In [12]:
%cd /content/PolyGen/third_party/TripoSR

!pip install -r requirements_remaining.txt

/content/PolyGen/third_party/TripoSR
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements_remaining.txt'


In [13]:
from pathlib import Path

# Locate the TripoSR repository
repo = Path("/content/PolyGen/third_party/TripoSR")

# Check original requirements file
requirements = repo / "requirements.txt"

if not requirements.exists():
    raise FileNotFoundError("Original requirements.txt is missing!")

# Read original dependencies
lines = requirements.read_text().splitlines()

# Exclude torchmcubes because we already installed it separately
remaining = [
    line for line in lines
    if "torchmcubes" not in line.lower()
]

# Create the new requirements file
output = repo / "requirements_remaining.txt"

output.write_text("\n".join(remaining) + "\n")

print("New requirements file created!")
print("File exists:", output.exists())
print("\nRemaining dependencies:")
print(output.read_text())

New requirements file created!
File exists: True

Remaining dependencies:
omegaconf==2.3.0
Pillow==10.1.0
einops==0.7.0
transformers==4.35.0
trimesh==4.0.5
rembg
huggingface-hub
imageio[ffmpeg]
gradio
xatlas==0.0.9
moderngl==5.10.0



In [15]:
%cd /content/PolyGen/third_party/TripoSR

!pip install -r requirements_remaining.txt

/content/PolyGen/third_party/TripoSR
  Using cached omegaconf-2.3.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached Pillow-10.1.0.tar.gz (50.8 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'Pillow' when getting requirements to build wheel


In [16]:
!pip install --only-binary=:all: "Pillow>=11.0.0"

In [17]:
from pathlib import Path

repo = Path("/content/PolyGen/third_party/TripoSR")

lines = (repo / "requirements.txt").read_text().splitlines()

remaining = []

for line in lines:
    clean = line.strip().lower()

    if "torchmcubes" in clean:
        continue

    if clean.startswith("pillow"):
        continue

    remaining.append(line)

output = repo / "requirements_remaining.txt"

output.write_text("\n".join(remaining) + "\n")

print("Updated requirements file created!")
print(output.read_text())

Updated requirements file created!
omegaconf==2.3.0
einops==0.7.0
transformers==4.35.0
trimesh==4.0.5
rembg
huggingface-hub
imageio[ffmpeg]
gradio
xatlas==0.0.9
moderngl==5.10.0



In [18]:
%cd /content/PolyGen/third_party/TripoSR

!pip install -r requirements_remaining.txt

/content/PolyGen/third_party/TripoSR
  Using cached omegaconf-2.3.0-py3-none-any.whl.metadata (3.9 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multip

In [19]:
from pathlib import Path

repo = Path("/content/PolyGen/third_party/TripoSR")

# Read original requirements
lines = (repo / "requirements.txt").read_text().splitlines()

# Packages already installed separately or postponed
skip_packages = [
    "torchmcubes",
    "pillow",
    "xatlas"
]

remaining = []

for line in lines:
    package = line.strip().lower()

    if any(
        package.startswith(name) or name in package
        for name in skip_packages
    ):
        continue

    remaining.append(line)

# Save a new requirements file
output = repo / "requirements_remaining.txt"

output.write_text("\n".join(remaining) + "\n")

print("Updated requirements file created!")
print("\nPackages postponed:")
print(skip_packages)

print("\nRemaining packages:")
print(output.read_text())

Updated requirements file created!

Packages postponed:
['torchmcubes', 'pillow', 'xatlas']

Remaining packages:
omegaconf==2.3.0
einops==0.7.0
transformers==4.35.0
trimesh==4.0.5
rembg
huggingface-hub
imageio[ffmpeg]
gradio
moderngl==5.10.0



In [20]:
%cd /content/PolyGen/third_party/TripoSR

!pip install -r requirements_remaining.txt

/content/PolyGen/third_party/TripoSR
  Using cached omegaconf-2.3.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached einops-0.7.0-py3-none-any.whl.metadata (13 kB)
  Using cached transformers-4.35.0-py3-none-any.whl.metadata (123 kB)
  Using cached trimesh-4.0.5-py3-none-any.whl.metadata (18 kB)
  Using cached rembg-2.0.88-py3-none-any.whl.metadata (30 kB)
  Using cached moderngl-5.10.0-cp313-cp313-linux_x86_64.whl
  Using cached antlr4_python3_runtime-4.9.3-py3-none-any.whl
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.14.1-cp313-cp313-linux_x86_64.whl
  Using cached glcontext-2.5.0-cp313-cp313-linux_x86_64.whl
  Using cached huggingface_hub-0.17.3-py3-none-any.whl.metadata (13 kB)
  Using cached pillow-13.0.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached pymatting-1.1.16-py3-none-any.whl.metadata (9.1 kB)
  Using cached scikit_image-0.26.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_

In [1]:
%cd /content/PolyGen/third_party/TripoSR

!pip install -r requirements_remaining.txt

/content/PolyGen/third_party/TripoSR


In [2]:
import torch
import torchmcubes
import rembg
import trimesh
import transformers
import omegaconf
from PIL import Image

print("All important imports successful!")

print("PyTorch:", torch.__version__)
print("Pillow:", Image.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



No onnxruntime backend found.
Please install rembg with CPU or GPU support:

    pip install "rembg[cpu]"  # for CPU
    pip install "rembg[gpu]"  # for NVIDIA/CUDA GPU

For more information, see: https://github.com/danielgatis/rembg#installation
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/rembg/bg.py", line 9, in <module>
    import onnxruntime as ort  # type: ignore[import-untyped]
    ^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'onnxruntime'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_15798/863653462.py", line 3, in <cell line: 0>
    import rembg
  File "/usr/local/lib/python3.13/dist-packages/rembg/__init__.py", line 11, in <module

TypeError: object of type 'NoneType' has no len()

In [1]:
import torch
import torchmcubes
import rembg
import trimesh
import transformers
import omegaconf
from PIL import Image

print("All important imports successful!")

print("PyTorch:", torch.__version__)
print("Pillow:", Image.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



No onnxruntime backend found.
Please install rembg with CPU or GPU support:

    pip install "rembg[cpu]"  # for CPU
    pip install "rembg[gpu]"  # for NVIDIA/CUDA GPU

For more information, see: https://github.com/danielgatis/rembg#installation
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/rembg/bg.py", line 9, in <module>
    import onnxruntime as ort  # type: ignore[import-untyped]
    ^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'onnxruntime'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_16546/863653462.py", line 3, in <cell line: 0>
    import rembg
  File "/usr/local/lib/python3.13/dist-packages/rembg/__init__.py", line 11, in <module

TypeError: object of type 'NoneType' has no len()

In [2]:
from pathlib import Path

repo = Path("/content/PolyGen/third_party/TripoSR")

# Read original requirements
lines = (repo / "requirements.txt").read_text().splitlines()

# Packages already installed separately or postponed
skip_packages = [
    "torchmcubes",
    "pillow",
    "xatlas"
]

remaining = []

for line in lines:
    package = line.strip().lower()

    if any(
        package.startswith(name) or name in package
        for name in skip_packages
    ):
        continue

    remaining.append(line)

# Save a new requirements file
output = repo / "requirements_remaining.txt"

output.write_text("\n".join(remaining) + "\n")

print("Updated requirements file created!")
print("\nPackages postponed:")
print(skip_packages)

print("\nRemaining packages:")
print(output.read_text())

Updated requirements file created!

Packages postponed:
['torchmcubes', 'pillow', 'xatlas']

Remaining packages:
omegaconf==2.3.0
einops==0.7.0
transformers==4.35.0
trimesh==4.0.5
rembg
huggingface-hub
imageio[ffmpeg]
gradio
moderngl==5.10.0



In [3]:
%cd /content/PolyGen/third_party/TripoSR

!pip install -r requirements_remaining.txt

/content/PolyGen/third_party/TripoSR


In [4]:
import torch
import torchmcubes
import rembg
import trimesh
import transformers
import omegaconf
from PIL import Image

print("All important imports successful!")

print("PyTorch:", torch.__version__)
print("Pillow:", Image.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



No onnxruntime backend found.
Please install rembg with CPU or GPU support:

    pip install "rembg[cpu]"  # for CPU
    pip install "rembg[gpu]"  # for NVIDIA/CUDA GPU

For more information, see: https://github.com/danielgatis/rembg#installation
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/rembg/bg.py", line 9, in <module>
    import onnxruntime as ort  # type: ignore[import-untyped]
    ^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'onnxruntime'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_16546/863653462.py", line 3, in <cell line: 0>
    import rembg
  File "/usr/local/lib/python3.13/dist-packages/rembg/__init__.py", line 11, in <module

TypeError: object of type 'NoneType' has no len()

In [5]:
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 58.8 MB/s  0:00:00


In [6]:
import onnxruntime as ort

print("ONNX Runtime installed:", ort.__version__)

print("Available providers:")
print(ort.get_available_providers())

ONNX Runtime installed: 1.32.0
Available providers:
['AzureExecutionProvider', 'CPUExecutionProvider']


In [7]:
import torch
import torchmcubes
import rembg
import trimesh
import transformers
import omegaconf
from PIL import Image

print("All important imports successful!")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

All important imports successful!
PyTorch: 2.14.1+cu130
CUDA: True
GPU: Tesla T4


/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [8]:
%cd /content/PolyGen/third_party/TripoSR

!python run.py --help

/content/PolyGen/third_party/TripoSR
Traceback (most recent call last):
  File "/content/PolyGen/third_party/TripoSR/run.py", line 9, in <module>
    import xatlas
ModuleNotFoundError: No module named 'xatlas'


In [9]:
!pip install --only-binary=:all: xatlas

In [10]:
import xatlas

print("xatlas installed successfully!")

xatlas installed successfully!


In [11]:
%cd /content/PolyGen/third_party/TripoSR

!python run.py --help

/content/PolyGen/third_party/TripoSR
usage: run.py [-h] [--device DEVICE]
              [--pretrained-model-name-or-path PRETRAINED_MODEL_NAME_OR_PATH]
              [--chunk-size CHUNK_SIZE] [--mc-resolution MC_RESOLUTION]
              [--no-remove-bg] [--foreground-ratio FOREGROUND_RATIO]
              [--output-dir OUTPUT_DIR] [--model-save-format {obj,glb}]
              [--bake-texture] [--texture-resolution TEXTURE_RESOLUTION]
              [--render]
              image [image ...]

positional arguments:
  image                 Path to input image(s).

options:
  -h, --help            show this help message and exit
  --device DEVICE       Device to use. If no CUDA-compatible device is found,
                        will fallback to 'cpu'. Default: 'cuda:0'
  --pretrained-model-name-or-path PRETRAINED_MODEL_NAME_OR_PATH
                        Path to the pretrained model. Could be either a
                        huggingface model id is or a local path. Default:
              

In [12]:
%cd /content/PolyGen/third_party/TripoSR

from pathlib import Path

image = Path("examples/chair.png")

print("Chair image exists:", image.exists())

if image.exists():
    print("Image size:", image.stat().st_size, "bytes")

/content/PolyGen/third_party/TripoSR
Chair image exists: True
Image size: 114705 bytes


In [13]:
%cd /content/PolyGen/third_party/TripoSR

!python run.py examples/chair.png --output-dir /content/PolyGen/outputs/triposr_test

/content/PolyGen/third_party/TripoSR
2026-11-26 17:47:30,792 - INFO - Initializing model ...
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2026-11-26 17:47:53,977 - INFO - Initializing model finished in 23185.33ms.
2026-11-26 17:47:53,977 - INFO - Processing images ...
100%|██████████████████

In [14]:
from pathlib import Path

output = Path("/content/PolyGen/outputs/triposr_test")

print("Generated files:")

for file in output.rglob("*"):
    if file.is_file():
        print(file.name, "-", round(file.stat().st_size / 1024, 2), "KB")

Generated files:
input.png - 91.19 KB
mesh.obj - 4443.05 KB


In [15]:
import trimesh
from pathlib import Path

mesh_path = Path(
    "/content/PolyGen/outputs/triposr_test/mesh.obj"
)

# Load generated 3D mesh
mesh = trimesh.load(
    str(mesh_path),
    force="mesh"
)

print("----- TRIPOSR MESH RESULTS -----")

print("Vertices:", len(mesh.vertices))
print("Triangles:", len(mesh.faces))

print("Bounds:")
print(mesh.bounds)

print("Watertight:", mesh.is_watertight)

print("Surface Area:", mesh.area)
print("Volume:", mesh.volume)

ValueError: string is not a file: /content/PolyGen/outputs/triposr_test/mesh.obj

In [16]:
import trimesh
from pathlib import Path

mesh_path = Path(
    "/content/PolyGen/outputs/triposr_test/mesh.obj"
)

# Load generated 3D mesh
mesh = trimesh.load(
    str(mesh_path),
    force="mesh"
)

print("----- TRIPOSR MESH RESULTS -----")

print("Vertices:", len(mesh.vertices))
print("Triangles:", len(mesh.faces))

print("Bounds:")
print(mesh.bounds)

print("Watertight:", mesh.is_watertight)

print("Surface Area:", mesh.area)
print("Volume:", mesh.volume)

ValueError: string is not a file: /content/PolyGen/outputs/triposr_test/mesh.obj

In [17]:
from pathlib import Path

base = Path("/content/PolyGen")

print("PolyGen folder exists:", base.exists())

print("\nSearching for OBJ files...\n")

if base.exists():
    for file in base.rglob("*.obj"):
        print(file, "-", round(file.stat().st_size / 1024, 2), "KB")

PolyGen folder exists: True

Searching for OBJ files...

/content/PolyGen/outputs/triposr_test/0/mesh.obj - 4443.05 KB


In [18]:
import trimesh
from pathlib import Path

base = Path("/content/PolyGen")

mesh_files = list(base.rglob("mesh.obj"))

if not mesh_files:
    print("mesh.obj not found. We need to generate it again.")
else:
    mesh_path = mesh_files[0]

    print("Loading:", mesh_path)

    mesh = trimesh.load(str(mesh_path), force="mesh")

    print("\n--- TRIPOSR MESH RESULTS ---")
    print("Vertices:", len(mesh.vertices))
    print("Triangles:", len(mesh.faces))
    print("Bounds:\n", mesh.bounds)
    print("Watertight:", mesh.is_watertight)
    print("Surface Area:", mesh.area)
    print("Volume:", mesh.volume)

Loading: /content/PolyGen/outputs/triposr_test/0/mesh.obj

--- TRIPOSR MESH RESULTS ---
Vertices: 42152
Triangles: 84158
Bounds:
 [[-0.31028777 -0.28045738 -0.49667114]
 [ 0.29341376  0.28813779  0.56946266]]
Watertight: False
Surface Area: 1.4883367253105493
Volume: 0.06451613802315123


In [19]:
%cd /content/PolyGen/third_party/TripoSR

!python run.py examples/chair.png --output-dir /content/PolyGen/outputs/triposr_test

/content/PolyGen/third_party/TripoSR
2026-11-26 17:56:16,402 - INFO - Initializing model ...
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2026-11-26 17:56:29,407 - INFO - Initializing model finished in 13004.82ms.
2026-11-26 17:56:29,407 - INFO - Processing images ...
2026-11-26 17:56:32,974

In [20]:
import trimesh
from pathlib import Path

mesh_path = Path(
    "/content/PolyGen/outputs/triposr_test/mesh.obj"
)

# Load generated 3D mesh
mesh = trimesh.load(
    str(mesh_path),
    force="mesh"
)

print("----- TRIPOSR MESH RESULTS -----")

print("Vertices:", len(mesh.vertices))
print("Triangles:", len(mesh.faces))

print("Bounds:")
print(mesh.bounds)

print("Watertight:", mesh.is_watertight)

print("Surface Area:", mesh.area)
print("Volume:", mesh.volume)

ValueError: string is not a file: /content/PolyGen/outputs/triposr_test/mesh.obj

In [21]:
from pathlib import Path

output = Path("/content/PolyGen/outputs/triposr_test")

print("Generated files:")

for file in output.rglob("*"):
    if file.is_file():
        print(
            file.name,
            round(file.stat().st_size / 1024, 2),
            "KB"
        )

Generated files:
input.png 91.19 KB
mesh.obj 4443.05 KB


In [22]:
import trimesh
from pathlib import Path

mesh_path = Path(
    "/content/PolyGen/outputs/triposr_test/mesh.obj"
)

mesh = trimesh.load(
    str(mesh_path),
    force="mesh"
)

print("===== TRIPOSR MESH REPORT =====")

print("Vertices:", len(mesh.vertices))
print("Triangles:", len(mesh.faces))

print("Bounds:")
print(mesh.bounds)

print("Watertight:", mesh.is_watertight)

print("Surface Area:", mesh.area)

print("Volume:", mesh.volume)

ValueError: string is not a file: /content/PolyGen/outputs/triposr_test/mesh.obj

In [23]:
from pathlib import Path

base = Path("/content/PolyGen")
output = base / "outputs" / "triposr_test"

print("Output folder exists:", output.exists())
print("\nFiles inside the output folder:")

if output.exists():
    for file in output.rglob("*"):
        if file.is_file():
            print(
                file.relative_to(base),
                "-",
                round(file.stat().st_size / 1024, 2),
                "KB"
            )

print("\nSearching for all OBJ files in PolyGen:")

for file in base.rglob("*.obj"):
    print(file)

Output folder exists: True

Files inside the output folder:
outputs/triposr_test/0/input.png - 91.19 KB
outputs/triposr_test/0/mesh.obj - 4443.05 KB

Searching for all OBJ files in PolyGen:
/content/PolyGen/outputs/triposr_test/0/mesh.obj


In [24]:
from pathlib import Path
import trimesh

output = Path("/content/PolyGen/outputs/triposr_test")

obj_files = list(output.rglob("*.obj"))

if not obj_files:
    print("No OBJ file found in TripoSR output.")
    print("Please share the Step 1 output.")
else:
    mesh_path = max(obj_files, key=lambda p: p.stat().st_size)

    print("Loading generated model:", mesh_path)

    mesh = trimesh.load(str(mesh_path), force="mesh")

    print("\n===== TRIPOSR MESH REPORT =====")
    print("Vertices:", len(mesh.vertices))
    print("Triangles:", len(mesh.faces))
    print("Bounds:\n", mesh.bounds)
    print("Watertight:", mesh.is_watertight)
    print("Surface Area:", mesh.area)
    print("Volume:", mesh.volume)

Loading generated model: /content/PolyGen/outputs/triposr_test/0/mesh.obj

===== TRIPOSR MESH REPORT =====
Vertices: 42152
Triangles: 84158
Bounds:
 [[-0.31028777 -0.28045738 -0.49667114]
 [ 0.29341376  0.28813779  0.56946266]]
Watertight: False
Surface Area: 1.4883367253105493
Volume: 0.06451613802315123


In [28]:
import trimesh
from pathlib import Path

output_dir = Path("/content/PolyGen/outputs/triposr_test")

# Dynamically search for any generated .obj file in the output folder
obj_files = list(output_dir.rglob("*.obj"))

if not obj_files:
    raise FileNotFoundError("No OBJ file was found in the output directory. Please verify that the generation step succeeded.")

# Select the first available mesh file
mesh_path = obj_files[0]
print(f"Loading generated model from: {mesh_path}")

mesh = trimesh.load(
    str(mesh_path),
    force="mesh"
)

print("===== TRIPOSR MESH REPORT =====")
print("Vertices:", len(mesh.vertices))
print("Triangles:", len(mesh.faces))
print("Bounds:")
print(mesh.bounds)
print("Watertight:", mesh.is_watertight)
print("Surface Area:", mesh.area)
print("Volume:", mesh.volume)

Loading generated model from: /content/PolyGen/outputs/triposr_test/0/mesh.obj
===== TRIPOSR MESH REPORT =====
Vertices: 42152
Triangles: 84158
Bounds:
[[-0.31028777 -0.28045738 -0.49667114]
 [ 0.29341376  0.28813779  0.56946266]]
Watertight: False
Surface Area: 1.4883367253105493
Volume: 0.06451613802315123


In [29]:
!nvidia-smi

Thu Nov 26 18:14:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             15W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [31]:
from google.colab import files
from pathlib import Path

output_dir = Path("/content/PolyGen/outputs/triposr_test")

# Search for any generated .obj file dynamically
obj_files = list(output_dir.rglob("*.obj"))

if not obj_files:
    raise FileNotFoundError("No OBJ file was found to download. Please verify that the generation step succeeded.")

target_file = obj_files[0]
print(f"Downloading: {target_file}")
files.download(str(target_file))

Downloading: /content/PolyGen/outputs/triposr_test/0/mesh.obj


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>